In [1]:
pip install tensorflow numpy pandas matplotlib opencv-python scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import cv2

# Dataset path
DATASET_PATH = r"C:\Users\rauna\Desktop\Main_Project\datasets"

# Image size
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# Load train dataset
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

# Load validation dataset
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

print("Class Mapping:", train_generator.class_indices)

Found 5933 images belonging to 2 classes.
Found 1483 images belonging to 2 classes.
Class Mapping: {'ALL blood': 0, 'normal  blood': 1}


In [3]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM, TimeDistributed, BatchNormalization

# Define CNN-LSTM Hybrid Model
model = Sequential([
    # CNN Feature Extractor
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),

    # Reshaping for LSTM
    Dense(256, activation='relu'),
    Dropout(0.5),
    
    # LSTM Layer
    tf.keras.layers.Reshape((1, 256)),  # Reshape for LSTM
    LSTM(128, return_sequences=False),

    # Output Layer
    Dense(1, activation='sigmoid')  # Binary Classification (Leukemia vs Normal)
])

# Compile Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()

C:\Users\rauna\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,714,177 (25.61 MB)

 Trainable params: 6,713,729 (25.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [4]:
# Train Model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20
)

# Save Model
model.save("binary.keras")


Epoch 1/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 312s 2s/step - accuracy: 0.9966 - loss: 0.0110 - val_accuracy: 0.9953 - val_loss: 0.0296
Epoch 2/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 90s 484ms/step - accuracy: 0.9997 - loss: 0.0017 - val_accuracy: 0.8159 - val_loss: 0.4492
Epoch 3/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 100s 536ms/step - accuracy: 0.9997 - loss: 0.0024 - val_accuracy: 0.5610 - val_loss: 3.7683
Epoch 4/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 119s 642ms/step - accuracy: 1.0000 - loss: 1.4795e-04 - val_accuracy: 0.9825 - val_loss: 0.0957
Epoch 5/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 111s 596ms/step - accuracy: 0.9983 - loss: 0.0061 - val_accuracy: 0.9906 - val_loss: 0.0311
Epoch 6/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 103s 551ms/step - accuracy: 0.9995 - loss: 0.0024 - val_accuracy: 0.7539 - val_loss: 1.8379
Epoch 7/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 100s 536ms/step - accuracy: 0.9997 - loss: 9.9482e-04 - val_accuracy: 0.9946 - val_loss: 0.0241
Epoch 8/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 100s 539ms/step - accuracy: 0.99

In [4]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np

# Load trained model
model = load_model("binary.keras")

# Load test image
def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (128, 128))
    img = img / 255.0  # Normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension

    prediction = model.predict(img)
    if prediction[0] < 0.5:
        print("Prediction: Leukemia detected")
    else:
        print("Prediction: Normal blood cell")

# Example Usage
predict_image(r"C:\Users\rauna\Desktop\Main_Project\datasets\normal  blood\ig\IG_5887.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
Prediction: Normal blood cell
